In [ ]:
import pandas as pd

In [ ]:
from pathlib import Path

# Path to the dataset (notebooks folder -> project root -> data/raw)
DATASET_DIR = Path("../data/raw")

# Check if the dataset exists
if DATASET_DIR.exists():
    print(f"Dataset found at: {DATASET_DIR.resolve()}")
else:
    print("Dataset not found.")

In [ ]:
# Load dataset labels/metadata
CSV_PATH = DATASET_DIR / "Data_Entry_2017.csv"

df = pd.read_csv(CSV_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

In [ ]:
# Check unique disease labels and their counts

label_counts = df["Finding Labels"].value_counts()

print("Number of unique labels:", len(label_counts))

label_counts.head(20)

In [ ]:
# Check for missing values

missing_values = df.isnull().sum()

print("Missing values per column:")
print(missing_values)

print("\nTotal missing values:", missing_values.sum())

In [ ]:
# Remove columns that are completely empty

df = df.dropna(axis=1, how="all")

print("Dataset cleaned successfully!")
print("New shape:", df.shape)

df.head()

In [ ]:
# Check if all images in the CSV exist in the image folder

IMAGE_DIR = DATASET_DIR / "images-224"

missing_images = []

for image_name in df["Image Index"]:
    image_path = IMAGE_DIR / image_name

    if not image_path.exists():
        missing_images.append(image_name)

print("Total images listed in CSV:", len(df))
print("Missing image files:", len(missing_images))

if len(missing_images) > 0:
    print("Examples of missing images:")
    print(missing_images[:10])
else:
    print("All images are available!")

In [ ]:
# Check the actual image folder contents

print("Image directory:", IMAGE_DIR.resolve())

# List first few items inside images-224
items = list(IMAGE_DIR.iterdir())

print("Number of items:", len(items))

print("First 10 items:")
for item in items[:10]:
    print(item)

In [ ]:
# Correct image directory path

IMAGE_DIR = DATASET_DIR / "images-224" / "images-224"

print("Updated image directory:", IMAGE_DIR.resolve())

# Check images
missing_images = []

for image_name in df["Image Index"]:
    image_path = IMAGE_DIR / image_name

    if not image_path.exists():
        missing_images.append(image_name)

print("Total images listed in CSV:", len(df))
print("Missing image files:", len(missing_images))

if len(missing_images) == 0:
    print("All images are available!")
else:
    print("Examples of missing images:")
    print(missing_images[:10])

In [ ]:
# Create binary classification labels: normal vs abnormal


def assign_label(finding):
    if finding == "No Finding":
        return "normal"
    else:
        return "abnormal"


df["Label"] = df["Finding Labels"].apply(assign_label)

# Check distribution
print(df["Label"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

# First split: 80% train, 20% temporary (val + test)
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["Label"], random_state=42)

# Second split: divide temporary into validation and test (10% each of total)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["Label"], random_state=42
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

print("\nTrain distribution:")
print(train_df["Label"].value_counts())

print("\nValidation distribution:")
print(val_df["Label"].value_counts())

print("\nTest distribution:")
print(test_df["Label"].value_counts())

In [ ]:
# Cell: copy_dataset_folders
import shutil

# Destination folders
OUTPUT_DIR = DATASET_DIR

splits = {
    "train": train_df,
    "val": val_df,
    "test": test_df,
}

# Create folders and copy images
for split_name, split_df in splits.items():
    for label in ["normal", "abnormal"]:
        # Create folder
        dest_dir = OUTPUT_DIR / split_name / label
        dest_dir.mkdir(parents=True, exist_ok=True)

        existing_images = list(dest_dir.glob("*.png"))
        if existing_images:
            print(f"Skipping {split_name}/{label}: {len(existing_images)} images already present")
            continue

        # Filter images
        label_df = split_df[split_df["Label"] == label]

        print(f"Copying {split_name}/{label}: {len(label_df)} images")

        for image_name in label_df["Image Index"]:
            src = IMAGE_DIR / image_name
            dst = dest_dir / image_name

            if not dst.exists():
                shutil.copy2(src, dst)

print("Dataset organization completed!")

In [ ]:
# Create metadata CSV

metadata = pd.concat(
    [train_df.assign(Split="train"), val_df.assign(Split="val"), test_df.assign(Split="test")]
)

# Add full image paths
metadata["File Path"] = metadata.apply(
    lambda row: (Path(row["Split"]) / row["Label"] / row["Image Index"]).as_posix(), axis=1
)

# Select useful columns
metadata = metadata[["File Path", "Image Index", "Finding Labels", "Label", "Split"]]

# Save metadata
METADATA_PATH = DATASET_DIR / "metadata.csv"

metadata.to_csv(METADATA_PATH, index=False)

print("Metadata created successfully!")
print("Saved at:", METADATA_PATH.resolve())

metadata.head()

In [ ]:
# Final dataset structure verification

from pathlib import Path

folders = [
    DATASET_DIR / "train" / "normal",
    DATASET_DIR / "train" / "abnormal",
    DATASET_DIR / "val" / "normal",
    DATASET_DIR / "val" / "abnormal",
    DATASET_DIR / "test" / "normal",
    DATASET_DIR / "test" / "abnormal",
]

print("Dataset folder verification:")

for folder in folders:
    count = len(list(folder.glob("*.png")))
    print(f"{folder.relative_to(DATASET_DIR)} : {count} images")

print("Metadata file exists:", (DATASET_DIR / "metadata.csv").exists())

print("Total images organized:", sum(len(list(folder.glob("*.png"))) for folder in folders))

In [ ]:
from pathlib import Path

# Define dataset path again
DATASET_DIR = Path("../data/raw")

print("Dataset directory:", DATASET_DIR.resolve())

In [ ]:
from pathlib import Path

DOC_PATH = DATASET_DIR / "dataset_info.md"

content = f"""# Chest X-Ray Dataset Information

## Dataset Source
NIH Chest X-ray Dataset (NIH Clinical Center)

## Dataset Description
The dataset contains 112,120 chest X-ray images with associated disease labels.

## Dataset Organization
- train/normal: {len(list((DATASET_DIR / "train" / "normal").glob("*.png")))} images
- train/abnormal: {len(list((DATASET_DIR / "train" / "abnormal").glob("*.png")))} images
- val/normal: {len(list((DATASET_DIR / "val" / "normal").glob("*.png")))} images
- val/abnormal: {len(list((DATASET_DIR / "val" / "abnormal").glob("*.png")))} images
- test/normal: {len(list((DATASET_DIR / "test" / "normal").glob("*.png")))} images
- test/abnormal: {len(list((DATASET_DIR / "test" / "abnormal").glob("*.png")))} images

## Files
- Labels CSV: {CSV_PATH}
- Metadata CSV: {METADATA_PATH}
"""

DOC_PATH.write_text(content, encoding="utf-8")
print(f"Dataset info file created at: {DOC_PATH.resolve()}")